In [1]:
import copy
import os
import argparse
from erasure.utils.logger import GLogger
import torch
import numpy as np
import random
from erasure.utils.config.local_ctx import Local
from erasure.utils.config.global_ctx import Global, bcolors 
from erasure.core.factory_base import ConfigurableFactory
from erasure.data.datasets.DatasetManager import DatasetManager

%load_ext autoreload
%autoreload 2

/NFSHOME/adangelo/miniconda3/envs/pytorch/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config_file = os.path.join("configs","LinkAttack", "Citeseer_edge.jsonc")

In [3]:
global_ctx = Global(config_file)
global_ctx.factory = ConfigurableFactory(global_ctx)

#Create Dataset
data_manager = global_ctx.factory.get_object( Local( global_ctx.config.data ))
global_ctx.dataset = data_manager

@,1596346903 | INFO | 851240 - Creating Global Context for: configs/LinkAttack/Citeseer_edge.jsonc


७
�,1596346929 | INFO | 851240 - Setting seeds to: 0
!,1596347078 | INFO | 851240 - REMOVAL TYPE SET AS edge
,1596347079 | INFO | 851240 - Caching System: False.
x�J6�,1596347203 | INFO | 851240 - Instantiating: torch_geometric.datasets.Planetoid
0�+�,1596347953 | INFO | 851240 - Created Configurable: erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource
,1596347954 | INFO | 851240 - {'class': 'erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource', 'parameters': {'datasource': {'class': 'torch_geometric.datasets.Planetoid', 'parameters': {'root': 'resources/data', 'name': 'Citeseer'}, 'preprocess': []}}}
x�J6�,1596348516 | INFO | 851240 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
0�?S�,1596348657 | INFO | 851240 - ['all', 'all_shuffled', '-']
x�J6�,1596348658 | INFO | 851240 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
����,1596348672 | INFO | 851240 - ['all', '

In [4]:
#Create Predictor
current = Local(global_ctx.config.predictor)
current.dataset = data_manager
predictor = global_ctx.factory.get_object(current)
global_ctx.predictor = predictor
global_ctx.logger.info('Global Predictor: ' + str(predictor))

KeyboardInterrupt: 

In [ ]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [ ]:
data_manager.partitions['all'][0][1]

tensor([3, 1, 5,  ..., 3, 1, 5])

In [ ]:
import torch
print(torch.version.cuda)

12.6


In [ ]:
print(predictor.optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.001
    lr: 0.0005000000000000002
    maximize: False
    weight_decay: 0
)


In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)


2.7.0+cu126
12.6


In [ ]:
#Create unlearners 
unlearners = []
unlearners_cfg = global_ctx.config.unlearners
for un in unlearners_cfg:
    current = Local(un)
    current.dataset = data_manager
    current.predictor = copy.deepcopy(predictor)
    unlearners.append( global_ctx.factory.get_object(current) )

�U��V,1491749847 | INFO | 330243 - Instantiating: torch.optim.Adam
�U&@U,1491749877 | INFO | 330243 - Created Configurable: erasure.unlearners.graph_unlearners.UNSIR.UNSIR


/NFSHOME/adangelo/LinkInference/erasure/unlearners/graph_unlearners/GraphUnlearner.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.labels = torch.tensor(self.labels)


In [ ]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [ ]:
data_manager.partitions['all'][0][0].edge_index

tensor([[   0,    1,    1,  ..., 3324, 3325, 3326],
        [ 628,  158,  486,  ..., 2820, 1643,   33]])

In [ ]:
data_manager.partitions['all'].revise_graph_edges([(0,1),(0,2)])[0][0]

Data(x=[3327, 3703], edge_index=[2, 0])

In [ ]:
print(len( data_manager.partitions['forget']) )

1820


In [ ]:
#Evaluator
current = Local(global_ctx.config.evaluator)
current.unlearners = unlearners
evaluator = global_ctx.factory.get_object(current)

# Evaluations
for unlearner in unlearners:
    global_ctx.logger.info(f'''{bcolors.OKGREEN}####\t\t Evaluating Unlearner {unlearner.__class__.__name__} \t\t####{bcolors.ENDC}''')
    evaluator.evaluate(unlearner,predictor)

�U&@U,1491750307 | INFO | 330243 - Created Configurable: erasure.evaluations.running.RunTime
�U&@U,1491750322 | INFO | 330243 - Created Configurable: erasure.unlearners.GoldModel.GoldModel
,1491750322 | INFO | 330243 - Unlearning copyed predictor: global


,1491750670 | INFO | 330243 - Loaded Instance from: resources/cached/erasure.model.graphs.GCN.GCN-99487ae94c2a1057e985b2699441d92c
�U&@U,1491750671 | INFO | 330243 - Created Configurable: erasure.model.TorchGraphModel.TorchGraphModel
�U&@U,1491750721 | INFO | 330243 - Created Configurable: erasure.evaluations.measures.AINGraph
�U&@U,1491750752 | INFO | 330243 - Created Configurable: erasure.evaluations.measures.RelearnTimeGraph
�U&@U,1491750753 | INFO | 330243 - Created Configurable: erasure.evaluations.measures.AUSGraph
�U��V,1491753534 | INFO | 330243 - Instantiating: torch.nn.CrossEntropyLoss
�U&@U,1491753535 | INFO | 330243 - Created Configurable: erasure.evaluations.MIA.graph_umia.Attack
�U&@U,1491753752 | INFO | 330243 - Created Configurable: erasure.evaluations.LinkTeller.LinkTeller.LinkTeller
�U&@U,1491753754 | INFO | 330243 - Created Configurable: erasure.evaluations.LinkTeller.LinkTeller.LinkTeller
�U&@U,1491753783 | INFO | 330243 - Created Configurable: erasure.eva

  7%|▋         | 14/187 [00:00<00:10, 16.58it/s]


KeyboardInterrupt: 